# CheXpert Robustness Extension — Complete Pipeline (3-Seed Ensembled Composite Model)

Single notebook: reloads all 3 seeds × 5 strategies (45 checkpoints), builds the composite model, then runs natural corruption sweep, Grad-CAM, adversarial (FGSM) testing, and cross-strategy robustness comparison — all in one run.

### Before running — in the Kaggle UI:
1. **Enable GPU**: Settings → Accelerator → GPU T4 x2
2. **Add the CheXpert dataset**: + Add Input → search "chexpert"
3. **Add Session 1 (seed-1)**, **Session 1b (seed-2)**, and **Session 1c (seed-3)** notebook outputs, all via + Add Input


### Locate dataset and all checkpoints (3 seeds)

In [ ]:
import os, glob

DATA_ROOT = None
for root, dirs, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("CheXpert dataset not found under /kaggle/input — add it via '+ Add Input'.")
print(f"Found dataset at: {DATA_ROOT}")

all_checkpoints = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
print(f"Found {len(all_checkpoints)} total checkpoint files (should be 45: 15 per seed x 3 seeds)")

def latest_checkpoint_for(prefix, tag):
    if tag == 'seed1':
        matches = [c for c in all_checkpoints if os.path.basename(c).startswith(prefix)
                   and '_seed2_' not in os.path.basename(c) and '_seed3_' not in os.path.basename(c)]
    else:
        marker = f'_{tag}_'
        matches = [c for c in all_checkpoints if os.path.basename(c).startswith(prefix) and marker in os.path.basename(c)]
    matches.sort(key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p).split('_e')[-1]))))
    if not matches:
        raise FileNotFoundError(f"No checkpoint found for prefix='{prefix}', tag={tag}")
    return matches[-1]

seed1_checkpoint_paths, seed2_checkpoint_paths, seed3_checkpoint_paths = {}, {}, {}
for strat, prefix in [('U-Zeros', 'zeros_e'), ('U-Ones', 'ones_e'), ('U-Ignore', 'ignore_e'),
                       ('U-SelfTrained', 'selftrained_e'), ('U-MultiClass', 'multiclass_e')]:
    seed1_checkpoint_paths[strat] = latest_checkpoint_for(prefix, 'seed1')
    seed2_checkpoint_paths[strat] = latest_checkpoint_for(prefix.replace('_e', '_seed2_e'), 'seed2')
    seed3_checkpoint_paths[strat] = latest_checkpoint_for(prefix.replace('_e', '_seed3_e'), 'seed3')

print("Seed-1:", seed1_checkpoint_paths)
print("Seed-2:", seed2_checkpoint_paths)
print("Seed-3:", seed3_checkpoint_paths)


### Rebuild dataset pipeline and model/eval helpers

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image

PATHOLOGIES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
BATCH_SIZE = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

val_transform = transforms.Compose([
    transforms.Resize((320, 320)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ImageLabelDataset(Dataset):
    def __init__(self, df, data_root, pathologies, transform, label_dtype=torch.float32):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.pathologies = pathologies
        self.transform = transform
        self.label_dtype = label_dtype
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['Path'].replace('CheXpert-v1.0-small/', '')
        img_path = f"{self.data_root}/{relative_path}"
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        np_dtype = np.int64 if self.label_dtype == torch.long else np.float32
        labels = torch.tensor(row[self.pathologies].values.astype(np_dtype), dtype=self.label_dtype)
        return image, labels

val_df = pd.read_csv(f'{DATA_ROOT}/valid.csv')
val_df = val_df[val_df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
for p in PATHOLOGIES:
    val_df[p] = val_df[p].fillna(0)

val_dataset = ImageLabelDataset(val_df, DATA_ROOT, PATHOLOGIES, val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Validation size: {len(val_dataset)}")

def build_model(num_classes=len(PATHOLOGIES)):
    model = models.densenet121(weights='DEFAULT')
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model.to(device)

def build_model_multiclass(num_classes=len(PATHOLOGIES)):
    model = models.densenet121(weights='DEFAULT')
    model.classifier = nn.Linear(model.classifier.in_features, num_classes * 3)
    return model.to(device)

def evaluate(model, loader, device, pathologies):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = torch.sigmoid(model(images))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())
    all_preds = np.concatenate(all_preds); all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs

os.makedirs('/kaggle/working/results', exist_ok=True)

def evaluate_multiclass(model, loader, device, pathologies):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).view(-1, len(pathologies), 3)
            logit_neg, logit_pos = outputs[:, :, 0], outputs[:, :, 1]
            p_pos = torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[..., 1]
            all_probs.append(p_pos.cpu().numpy())
            all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs); all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_probs[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs


### Rebuild 3-seed ensembles and the final composite model

In [ ]:
def load_seed(strat, path, multiclass=False):
    m = build_model_multiclass() if multiclass else build_model()
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

models_by_seed = {'seed1': seed1_checkpoint_paths, 'seed2': seed2_checkpoint_paths, 'seed3': seed3_checkpoint_paths}
loaded = {s: {} for s in models_by_seed}
for seed_tag, paths in models_by_seed.items():
    for strat in ['U-Zeros', 'U-Ones', 'U-Ignore', 'U-SelfTrained', 'U-MultiClass']:
        loaded[seed_tag][strat] = load_seed(strat, paths[strat], multiclass=(strat == 'U-MultiClass'))

class SeedEnsembleModel(nn.Module):
    """Averages predicted probabilities across N independently-trained seeds of the same strategy."""
    def __init__(self, models_list, is_multiclass=False, num_pathologies=len(PATHOLOGIES)):
        super().__init__()
        self.models_list = nn.ModuleList(models_list)
        self.is_multiclass = is_multiclass
        self.num_pathologies = num_pathologies

    def _to_probs(self, model, x):
        out = model(x)
        if self.is_multiclass:
            out = out.view(-1, self.num_pathologies, 3)
            logit_neg, logit_pos = out[:, :, 0], out[:, :, 1]
            return torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[..., 1]
        return torch.sigmoid(out)

    def forward(self, x):
        probs = [self._to_probs(m, x) for m in self.models_list]
        p_avg = torch.stack(probs, dim=0).mean(dim=0).clamp(1e-6, 1 - 1e-6)
        return torch.log(p_avg / (1 - p_avg))

strategy_models = {}
for strat in ['U-Zeros', 'U-Ones', 'U-Ignore', 'U-SelfTrained', 'U-MultiClass']:
    is_mc = (strat == 'U-MultiClass')
    strategy_models[strat] = SeedEnsembleModel(
        [loaded['seed1'][strat], loaded['seed2'][strat], loaded['seed3'][strat]], is_multiclass=is_mc
    ).to(device)

strategy_aucs = {name: evaluate(model, val_loader, device, PATHOLOGIES) for name, model in strategy_models.items()}
comparison_df = pd.DataFrame(strategy_aucs).T[PATHOLOGIES]
print("=== 3-Seed Ensembled AUC by Strategy and Pathology ===")
print(comparison_df.round(4).to_string())

best_strategy_per_pathology = comparison_df.idxmax(axis=0)
print("\n=== Best Strategy per Pathology ===")
for p in PATHOLOGIES:
    print(f"  {p}: {best_strategy_per_pathology[p]} (AUC = {comparison_df.loc[best_strategy_per_pathology[p], p]:.4f})")

best_per_pathology = {}
_key_map = {}
for i, p in enumerate(PATHOLOGIES):
    strat = best_strategy_per_pathology[p]
    key = f"{strat}_{i}"
    best_per_pathology[p] = (strat, strategy_models[strat])
    _key_map[p] = key

class CompositeBestModel(nn.Module):
    def __init__(self, best_per_pathology, pathologies, key_map):
        super().__init__()
        self.pathologies = pathologies
        self.best_per_pathology = best_per_pathology
        self.key_map = key_map
        self.unique_models = nn.ModuleDict()
        for p in pathologies:
            strat, m = best_per_pathology[p]
            self.unique_models[key_map[p]] = m

    def forward(self, x):
        batch = x.size(0)
        final_logits = torch.zeros(batch, len(self.pathologies), device=x.device)
        cache = {}
        for i, p in enumerate(self.pathologies):
            key = self.key_map[p]
            if key not in cache:
                cache[key] = self.unique_models[key](x)
            final_logits[:, i] = cache[key][:, i]
        return final_logits

composite_model = CompositeBestModel(best_per_pathology, PATHOLOGIES, _key_map).to(device)
composite_model.eval()

composite_aucs = evaluate(composite_model, val_loader, device, PATHOLOGIES)
print("\n=== Final 3-Seed Composite Model AUCs ===")
for p, a in composite_aucs.items():
    print(f"  {p}: {a:.4f}")

comparison_df.loc['Composite (3-seed final)'] = pd.Series(composite_aucs)
comparison_df.to_csv('/kaggle/working/results/strategy_comparison.csv')


## EXTENSION: Robustness to Real-World Image Degradation

We test whether the **final composite model** (the paper-style best-strategy-per-pathology model built above) holds up under realistic image-quality degradation — blur, compression, brightness/contrast shifts, rotation — simulating conditions like low-resolution phone photos of X-rays or scans from lower-resource clinical settings, rather than the clean hospital-grade images the model was trained on.

**Hypothesis:** pathologies that depend on fine textural/density detail (e.g. Consolidation, Atelectasis) will degrade faster under blur/compression than pathologies with strong large-scale shape signal (e.g. Cardiomegaly).


### E1. Reference model for robustness testing

We use the `composite_model` built in Part 1 — the actual paper-style final model — as the reference for every robustness experiment below, rather than a single strategy's checkpoint. This means our robustness findings reflect the model you'd actually deploy, not an arbitrary single-strategy choice.

In [ ]:
robustness_model = composite_model
robustness_model.eval()
print("Using composite (best-per-pathology) model for robustness testing.")


### E2. Corruption functions

Eight corruption types, each with 5 severity levels (0 = none, 4 = severe), applied *before* resizing/normalization:
- **blur, jpeg, brightness, contrast, rotation** — general image-quality degradation
- **gaussian_noise** — sensor/transmission noise
- **downscale** — simulates a genuinely low-resolution capture (e.g. an old phone camera), not just blur
- **occlusion** — a random black patch, simulating something physically blocking part of the image (a wire, a sticker, a finger over the lens)


In [ ]:
import io
from PIL import ImageFilter, ImageEnhance
import random

def corrupt_blur(img, severity):
    if severity == 0:
        return img
    radius = [0, 1, 2, 4, 6][severity]
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def corrupt_jpeg(img, severity):
    if severity == 0:
        return img
    quality = [100, 80, 60, 40, 20][severity]
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

def corrupt_brightness(img, severity):
    if severity == 0:
        return img
    factor = [1.0, 0.8, 0.6, 0.4, 0.25][severity]
    return ImageEnhance.Brightness(img).enhance(factor)

def corrupt_contrast(img, severity):
    if severity == 0:
        return img
    factor = [1.0, 0.75, 0.55, 0.35, 0.2][severity]
    return ImageEnhance.Contrast(img).enhance(factor)

def corrupt_rotation(img, severity, seed=None):
    if severity == 0:
        return img
    max_angle = [0, 3, 7, 12, 20][severity]
    rng = random.Random(seed)
    angle = rng.uniform(-max_angle, max_angle)
    return img.rotate(angle, fillcolor=(0, 0, 0))

def corrupt_gaussian_noise(img, severity):
    if severity == 0:
        return img
    std = [0, 8, 16, 28, 45][severity]
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, std, arr.shape)
    noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def corrupt_downscale(img, severity):
    if severity == 0:
        return img
    factor = [1, 4, 8, 16, 24][severity]  # simulates a low-res phone photo, upscaled back
    w, h = img.size
    small = img.resize((max(1, w // factor), max(1, h // factor)), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)

def corrupt_occlusion(img, severity, seed=None):
    if severity == 0:
        return img
    frac = [0, 0.08, 0.15, 0.25, 0.35][severity]  # fraction of image width/height occluded
    rng = random.Random(seed)
    img = img.copy()
    w, h = img.size
    patch_w, patch_h = int(w * frac), int(h * frac)
    x0 = rng.randint(0, max(1, w - patch_w))
    y0 = rng.randint(0, max(1, h - patch_h))
    arr = np.array(img)
    arr[y0:y0 + patch_h, x0:x0 + patch_w] = 0
    return Image.fromarray(arr)

CORRUPTIONS = {
    'blur': corrupt_blur,
    'jpeg': corrupt_jpeg,
    'brightness': corrupt_brightness,
    'contrast': corrupt_contrast,
    'rotation': corrupt_rotation,
    'gaussian_noise': corrupt_gaussian_noise,
    'downscale': corrupt_downscale,
    'occlusion': corrupt_occlusion,
}
SEVERITY_LEVELS = [0, 1, 2, 3, 4]


### E3. Corrupted dataset wrapper

In [ ]:
class CorruptedCheXpertDataset(Dataset):
    def __init__(self, base_df, data_root, pathologies, corruption_fn, severity, seed=None):
        self.df = base_df
        self.data_root = data_root
        self.pathologies = pathologies
        self.corruption_fn = corruption_fn
        self.severity = severity
        self.seed = seed
        self.final_transform = transforms.Compose([
            transforms.Resize((320, 320)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['Path'].replace('CheXpert-v1.0-small/', '')
        img_path = f"{self.data_root}/{relative_path}"
        image = Image.open(img_path).convert('RGB')

        if self.corruption_fn.__name__ in ('corrupt_rotation', 'corrupt_occlusion'):
            image = self.corruption_fn(image, self.severity, seed=self.seed)
        else:
            image = self.corruption_fn(image, self.severity)

        image = self.final_transform(image)
        labels = torch.tensor(row[self.pathologies].values.astype(np.float32))
        return image, labels


### E4. Run the multi-severity, multi-trial corruption sweep

For each corruption type and severity level, we run evaluation **3 times** (different random trial seeds) to get mean ± std AUC per pathology.

In [ ]:
N_TRIALS = 3
results_rows = []

print("Evaluating clean baseline...")
clean_dataset = CorruptedCheXpertDataset(val_df, DATA_ROOT, PATHOLOGIES, corrupt_blur, severity=0)
clean_loader = DataLoader(clean_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
clean_aucs = evaluate(robustness_model, clean_loader, device, PATHOLOGIES)
for p, auc in clean_aucs.items():
    results_rows.append({'corruption': 'clean', 'severity': 0, 'trial': 0, 'pathology': p, 'auc': auc})

for corruption_name, corruption_fn in CORRUPTIONS.items():
    for severity in SEVERITY_LEVELS[1:]:
        for trial in range(N_TRIALS):
            print(f"Evaluating {corruption_name} severity={severity} trial={trial}...")
            corrupted_dataset = CorruptedCheXpertDataset(val_df, DATA_ROOT, PATHOLOGIES, corruption_fn, severity, seed=trial)
            corrupted_loader = DataLoader(corrupted_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
            aucs = evaluate(robustness_model, corrupted_loader, device, PATHOLOGIES)
            for p, auc in aucs.items():
                results_rows.append({'corruption': corruption_name, 'severity': severity, 'trial': trial, 'pathology': p, 'auc': auc})

robustness_df = pd.DataFrame(results_rows)
robustness_df.to_csv('/kaggle/working/results/robustness_results.csv', index=False)
print("Done. Results saved.")
robustness_df.head(20)


### E5. Degradation curves — AUC vs. severity, per corruption type, per pathology

In [ ]:
import matplotlib.pyplot as plt

summary = robustness_df.groupby(['corruption', 'severity', 'pathology'])['auc'].agg(['mean', 'std']).reset_index()

clean_rows = summary[summary['corruption'] == 'clean'].copy()
for corruption_name in CORRUPTIONS.keys():
    temp = clean_rows.copy()
    temp['corruption'] = corruption_name
    summary = pd.concat([summary, temp], ignore_index=True)
summary = summary[summary['corruption'] != 'clean'].drop_duplicates(subset=['corruption', 'severity', 'pathology'])

fig, axes = plt.subplots(1, len(PATHOLOGIES), figsize=(24, 5), sharey=True)

for i, pathology in enumerate(PATHOLOGIES):
    ax = axes[i]
    path_data = summary[summary['pathology'] == pathology]
    for corruption_name in CORRUPTIONS.keys():
        c_data = path_data[path_data['corruption'] == corruption_name].sort_values('severity')
        ax.plot(c_data['severity'], c_data['mean'], marker='o', label=corruption_name)
        ax.fill_between(c_data['severity'],
                         c_data['mean'] - c_data['std'].fillna(0),
                         c_data['mean'] + c_data['std'].fillna(0),
                         alpha=0.15)
    ax.set_title(pathology)
    ax.set_xlabel('Severity')
    if i == 0:
        ax.set_ylabel('AUC')
    ax.set_ylim(0.4, 1.0)
    ax.grid(alpha=0.3)

axes[0].legend(loc='lower left', fontsize=8)
plt.suptitle('AUC Degradation Under Increasing Corruption Severity, by Pathology (Composite Model)', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/results/degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()


### E6. Per-pathology sensitivity ranking

In [ ]:
sensitivity_rows = []
for pathology in PATHOLOGIES:
    clean_auc = summary[(summary['pathology'] == pathology) & (summary['severity'] == 0)]['mean'].values
    clean_auc = clean_auc[0] if len(clean_auc) > 0 else clean_aucs[pathology]

    drops = []
    for corruption_name in CORRUPTIONS.keys():
        worst = summary[(summary['pathology'] == pathology) &
                         (summary['corruption'] == corruption_name) &
                         (summary['severity'] == max(SEVERITY_LEVELS))]['mean'].values
        if len(worst) > 0:
            drops.append(clean_auc - worst[0])

    sensitivity_rows.append({
        'pathology': pathology,
        'clean_auc': clean_auc,
        'avg_auc_drop': np.mean(drops),
        'max_auc_drop': np.max(drops)
    })

sensitivity_df = pd.DataFrame(sensitivity_rows).sort_values('avg_auc_drop', ascending=False)
print("Pathology sensitivity to corruption (higher avg_auc_drop = more fragile):\n")
print(sensitivity_df.to_string(index=False))
sensitivity_df.to_csv('/kaggle/working/results/sensitivity_ranking.csv', index=False)


### E7. Grad-CAM attention drift under increasing corruption

Since the composite model routes each pathology to a different underlying strategy model, Grad-CAM is run on **the specific model actually responsible for that pathology's prediction** — this is more correct than visualizing an arbitrary single model, since it shows exactly what the deployed composite is "looking at."

In [ ]:
!pip install -q grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

class MulticlassPositiveTarget:
    """Custom Grad-CAM target for the U-MultiClass model: uses (logit_pos - logit_neg) for the
    given pathology as a monotonic proxy for the positive-class probability."""
    def __init__(self, pathology_idx, num_pathologies):
        self.pathology_idx = pathology_idx
        self.num_pathologies = num_pathologies

    def __call__(self, model_output):
        reshaped = model_output.view(self.num_pathologies, 3)
        return reshaped[self.pathology_idx, 1] - reshaped[self.pathology_idx, 0]

def gradcam_at_severity(base_df, idx, pathology_name, corruption_fn=corrupt_blur, severities=(0, 1, 2, 3, 4)):
    # Grad-CAM runs on seed-1 of the winning strategy's 3-seed ensemble (representative single network).
    pathology_idx = PATHOLOGIES.index(pathology_name)
    strategy_name, ensemble_submodel = best_per_pathology[pathology_name]
    representative_submodel = ensemble_submodel.models_list[0]
    representative_submodel.eval()

    target_layer = [representative_submodel.features.denseblock4.denselayer16.conv2]
    cam = GradCAM(model=representative_submodel, target_layers=target_layer)

    if strategy_name == 'U-MultiClass':
        targets = [MulticlassPositiveTarget(pathology_idx, len(PATHOLOGIES))]
    else:
        targets = [ClassifierOutputTarget(pathology_idx)]

    fig, axes = plt.subplots(1, len(severities), figsize=(4 * len(severities), 4))

    for col, severity in enumerate(severities):
        ds = CorruptedCheXpertDataset(base_df, DATA_ROOT, PATHOLOGIES, corruption_fn, severity)
        img_tensor, labels = ds[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)

        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
        img_np = np.clip(std * img_np + mean, 0, 1)

        visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        axes[col].imshow(visualization)
        axes[col].set_title(f"Severity {severity}")
        axes[col].axis('off')

    fig.suptitle(f"Grad-CAM Attention Drift Under {corruption_fn.__name__}: {pathology_name} (via {strategy_name} model)", fontsize=14)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/results/gradcam_drift_{pathology_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

gradcam_at_severity(val_df, idx=0, pathology_name='Cardiomegaly', corruption_fn=corrupt_blur)

most_fragile = sensitivity_df.iloc[0]['pathology']
gradcam_at_severity(val_df, idx=0, pathology_name=most_fragile, corruption_fn=corrupt_blur)


### E9. Adversarial robustness (FGSM)

Natural corruptions (blur, noise, etc.) are one kind of robustness question. **Adversarial robustness** is a different, stricter one: how much does the model's prediction change under a *tiny, deliberately-crafted* pixel perturbation — small enough to be invisible to a human — computed via the Fast Gradient Sign Method (FGSM)?

**Memory note:** we attack only the specific strategy ensemble responsible for each pathology (not the full 15-network composite) — forcing gradients through all 5 strategies × 3 seeds simultaneously is unnecessary and exceeds GPU memory. This is also more methodologically correct: an adversarial attack on a given pathology's prediction should target the actual model producing that prediction.


In [ ]:
def fgsm_attack(model, images, labels, epsilon, pathology_idx):
    """Generates an adversarial perturbation for a single pathology's prediction and returns the perturbed images."""
    images = images.clone().detach().to(device).requires_grad_(True)
    outputs = model(images)
    target = labels[:, pathology_idx].to(device)
    loss = F.binary_cross_entropy_with_logits(outputs[:, pathology_idx], target)
    model.zero_grad()
    loss.backward()
    perturbation = epsilon * images.grad.sign()
    perturbed = (images + perturbation).clamp(-3, 3).detach()
    return perturbed

adv_epsilons = [0.0, 0.005, 0.01, 0.02, 0.04]
adv_results = []

# Use a smaller batch size here specifically -- FGSM needs a full gradient graph in memory,
# which is much more memory-hungry than plain no_grad() evaluation.
ADV_BATCH_SIZE = 8
adv_loader = DataLoader(val_dataset, batch_size=ADV_BATCH_SIZE, shuffle=False, num_workers=2)

for pathology_idx, pathology_name in enumerate(PATHOLOGIES):
    # IMPORTANT: attack only the specific strategy's ensemble responsible for this pathology,
    # not the full composite -- forcing gradients through all 5 strategies x 3 seeds (15 networks)
    # at once is what caused the earlier CUDA OutOfMemoryError. Attacking the actual model
    # responsible for this pathology's prediction is both correct and far cheaper.
    _, target_model = best_per_pathology[pathology_name]
    target_model.eval()

    for eps in adv_epsilons:
        all_preds, all_labels = [], []
        for images, labels in adv_loader:
            if eps == 0.0:
                perturbed = images.to(device)
            else:
                perturbed = fgsm_attack(target_model, images, labels, eps, pathology_idx)
            with torch.no_grad():
                preds = torch.sigmoid(target_model(perturbed))
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
            torch.cuda.empty_cache()
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        try:
            auc = roc_auc_score(all_labels[:, pathology_idx], all_preds[:, pathology_idx])
        except ValueError:
            auc = float('nan')
        adv_results.append({'pathology': pathology_name, 'epsilon': eps, 'auc': auc})
        print(f"{pathology_name} | epsilon={eps:.3f} | AUC={auc:.4f}")
    torch.cuda.empty_cache()

adv_df = pd.DataFrame(adv_results)
adv_df.to_csv('/kaggle/working/results/adversarial_results.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for pathology_name in PATHOLOGIES:
    sub = adv_df[adv_df['pathology'] == pathology_name]
    ax.plot(sub['epsilon'], sub['auc'], marker='o', label=pathology_name)
ax.set_xlabel('FGSM epsilon (perturbation size)')
ax.set_ylabel('AUC')
ax.set_title('Adversarial Robustness: AUC vs. FGSM Perturbation Strength')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/results/adversarial_curve.png', dpi=150, bbox_inches='tight')
plt.show()


### E10. Cross-strategy robustness comparison

Everything above tests the **composite** model. Here's a genuinely open question the paper never asks: **does the choice of uncertainty-handling strategy affect robustness, independent of clean-data accuracy?** A strategy might win on clean validation AUC but turn out to be more fragile under real-world degradation — or vice versa. We test all 5 individually-trained strategy models (not the composite) on a reduced corruption grid to keep this tractable, and compare average robustness across strategies.

In [ ]:
# Reduced grid to keep this tractable: 3 representative corruption types, 3 severities, 1 trial each
CROSS_STRATEGY_CORRUPTIONS = {'blur': corrupt_blur, 'gaussian_noise': corrupt_gaussian_noise, 'downscale': corrupt_downscale}
CROSS_STRATEGY_SEVERITIES = [0, 2, 4]

cross_strategy_results = []

for strategy_name, strat_model in strategy_models.items():
    strat_model.eval()
    # NOTE: strategy_models entries are SeedEnsembleModel wrappers (built during setup), which already
    # convert ALL strategies -- including U-MultiClass -- into a unified (batch, num_pathologies) logit
    # output internally. So every strategy uses the standard evaluate() function; no special-casing needed.

    for corruption_name, corruption_fn in CROSS_STRATEGY_CORRUPTIONS.items():
        for severity in CROSS_STRATEGY_SEVERITIES:
            ds = CorruptedCheXpertDataset(val_df, DATA_ROOT, PATHOLOGIES, corruption_fn, severity)
            loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
            aucs = evaluate(strat_model, loader, device, PATHOLOGIES)
            avg_auc = np.nanmean(list(aucs.values()))
            cross_strategy_results.append({
                'strategy': strategy_name, 'corruption': corruption_name,
                'severity': severity, 'avg_auc': avg_auc
            })
            print(f"[{strategy_name}] {corruption_name} sev={severity}: avg AUC = {avg_auc:.4f}")

cross_strategy_df = pd.DataFrame(cross_strategy_results)
cross_strategy_df.to_csv('/kaggle/working/results/cross_strategy_robustness.csv', index=False)

robustness_summary = []
for strategy_name in strategy_models.keys():
    strat_data = cross_strategy_df[cross_strategy_df['strategy'] == strategy_name]
    drops = []
    for corruption_name in CROSS_STRATEGY_CORRUPTIONS.keys():
        c_data = strat_data[strat_data['corruption'] == corruption_name]
        clean_val = c_data[c_data['severity'] == 0]['avg_auc'].values
        worst_val = c_data[c_data['severity'] == max(CROSS_STRATEGY_SEVERITIES)]['avg_auc'].values
        if len(clean_val) and len(worst_val):
            drops.append(clean_val[0] - worst_val[0])
    robustness_summary.append({'strategy': strategy_name, 'avg_robustness_drop': np.mean(drops)})

robustness_summary_df = pd.DataFrame(robustness_summary).sort_values('avg_robustness_drop')
print("\n=== Strategy Robustness Ranking (lower drop = more robust) ===")
print(robustness_summary_df.to_string(index=False))
robustness_summary_df.to_csv('/kaggle/working/results/strategy_robustness_ranking.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(robustness_summary_df['strategy'], robustness_summary_df['avg_robustness_drop'], color='steelblue')
ax.set_xlabel('Avg AUC Drop Under Corruption (lower = more robust)')
ax.set_title('Robustness Comparison Across Uncertainty-Handling Strategies')
plt.tight_layout()
plt.savefig('/kaggle/working/results/strategy_robustness_ranking.png', dpi=150, bbox_inches='tight')
plt.show()


### Final summary tables for your report

In [ ]:
print("=== Table A: Strategy Comparison (3-seed ensembled, AUC per pathology) ===")
print(comparison_df.round(4).to_string())

report_table = sensitivity_df.copy()
report_table['worst_case_auc'] = report_table['clean_auc'] - report_table['max_auc_drop']
report_table = report_table[['pathology', 'clean_auc', 'worst_case_auc', 'avg_auc_drop', 'max_auc_drop']]
report_table.columns = ['Pathology', 'Clean AUC', 'Worst-Case AUC', 'Avg AUC Drop', 'Max AUC Drop']
report_table = report_table.round(4)
print("\n=== Table B: Natural Corruption Robustness Summary (Composite Model) ===")
print(report_table.to_string(index=False))
report_table.to_csv('/kaggle/working/results/report_summary_table.csv', index=False)

print("\n=== Table C: Adversarial Robustness (FGSM) ===")
print(adv_df.round(4).to_string(index=False))

print("\n=== Table D: Cross-Strategy Robustness Ranking ===")
print(robustness_summary_df.round(4).to_string(index=False))


## ✅ Complete — Save Your Progress

Click **Save Version → Save & Run All (Commit)** to save all results permanently.